# WaterSoftHack 2026 — Day 3: Hybrid Pipeline (Hands-On)

**The finale.** Nothing in this notebook is new. Tuesday you trained a cloud now-cast; Wednesday you
programmed a smart gauge. Today you wire them into one early-warning system and watch it beat both
halves working alone.

The system, end to end:

1. **A fleet** of five smart gauges strung down a basin, with a storm rolling through them in sequence.
2. **A cloud brain** that ingests their messages and keeps a live basin picture.
3. **Trigger & burst:** the upstream gauge's alert makes the cloud forecast *every downstream reach* at once.
4. **The loop closes:** the cloud pushes **storm mode** back down, and we measure the extra warning time it buys.

> Same JupyterHub, same login. Run each cell with **Shift+Enter**. The slogan of the day:
> **let each layer do what only it can do.**

> **Concept - let each layer do what only it can do.** You built two half-systems this week. The cloud has the basin-wide view and can forecast hours ahead, but it dies when the storm cuts the link. The edge is autonomous and instant, but a single gauge knows only its own reach and cannot see the future. Their weaknesses are mirror images, which is exactly what makes a **hybrid** system work: the edge owns the instant and the outage (detect, alarm, endure), the cloud owns the basin and the future (fuse, forecast, decide at scale), and the wire between them is a **two-way street** - insight flows up (alerts, summaries) and guidance flows down (storm mode, updated models). Today is not new machinery, it is wiring the two halves together so each covers the other's blind spot.

---
## Part 0 — The basin and the storm

Five gauges down one river, from headwaters (gauge 0) to the town we are protecting (gauge 4). A flash
flood begins upstream and travels downstream, hitting each gauge about **90 minutes** after the one above
it. That travel time is the whole opportunity: if gauge 0 detects the rise, the town at gauge 4 has hours.

**The travelling storm.** Every gauge gets the same design storm, delayed downstream. Gauge $g$ is offset by $\Delta_g=g\,\tau$ with per-hop travel time $\tau=90$ min, so its rise begins at $t_r=1440\,d_{\text{storm}}+360+\Delta_g$ minutes and it crests at $t_p=t_r+360$. In minute index $t$ (with $n=1440\,\text{days}$) the stage is

$$s_g(t)=\operatorname{interp}\big(t;\ \{t_k\},\{s_k\}\big)+0.05\sin\tfrac{2\pi t}{1440}+\eta,\qquad \eta\sim\mathcal{N}(0,\,0.01^{2}),$$

over knot heights $s_k=[2,\,2,\,5,\,13,\,9,\,3,\,2.5]$ ft at times $t_k=[\,0,\ t_r,\ t_r{+}120,\ t_p,\ t_p{+}180,\ t_p{+}1500,\ n\,]$. The delay $\Delta_g$ is exactly what makes the flood wave reach each gauge in sequence.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_GAUGES     = 5
TRAVEL_MIN   = 90          # minutes the flood wave takes between adjacent gauges
FLOOD_STAGE  = 10.0        # ft
DAYS         = 14

def storm_at(offset_min, days=DAYS, storm_day=10, seed=0):
    """One gauge's minute-resolution stage record; the flood is delayed by offset_min."""
    n = days * 24 * 60
    t = np.arange(n)
    rise = storm_day * 24 * 60 + 6 * 60 + offset_min      # rise begins, shifted downstream
    peak = rise + 6 * 60
    keys_t = [0,    rise, rise + 120, peak, peak + 180, peak + 25*60, n]
    keys_v = [2.0,  2.0,  5.0,        13.0, 9.0,        3.0,          2.5]
    stage = np.interp(t, keys_t, keys_v)
    rng = np.random.default_rng(seed)
    stage += 0.05 * np.sin(2 * np.pi * t / (24*60)) + rng.normal(0, 0.01, n)
    idx = pd.date_range("2026-07-06", periods=n, freq="1min")
    return pd.Series(stage, index=idx, name="stage_ft")

basin = {g: storm_at(offset_min=g * TRAVEL_MIN, seed=g) for g in range(N_GAUGES)}
TIME = basin[0].index

fig, ax = plt.subplots(figsize=(11.5, 4.4))
for g in range(N_GAUGES):
    lbl = "gauge 0 (headwaters)" if g == 0 else ("gauge 4 (the town)" if g == N_GAUGES-1 else f"gauge {g}")
    ax.plot(basin[g].index, basin[g].values, lw=1.6, label=lbl)
ax.axhline(FLOOD_STAGE, color="crimson", ls="--", lw=1.3, label="Flood stage")
lo = pd.Timestamp("2026-07-16 00:00"); hi = pd.Timestamp("2026-07-17 12:00")
ax.set_xlim(lo, hi); ax.set_ylim(1.5, 13.5)
ax.set_title("One flood, five gauges - the wave travels downstream")
ax.set_ylabel("Stage (ft)"); ax.legend(fontsize=9, loc="upper right"); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

print("The same flood reaches the town about",
      round(N_GAUGES * TRAVEL_MIN / 60, 1), "hours after it hits the headwaters.")

**That downstream lag is the prize.** A purely local warning at the town (gauge 4) fires when the
water is already there. A *hybrid* system uses gauge 0's alarm plus a cloud forecast to warn the town
while its river is still quiet. Let's build it.

---
## Part 1 — The fleet (Wednesday's gauge, x5)

Here is the smart gauge from yesterday, trimmed to the essentials: adaptive sampling, the two detection
rules, hourly summaries, and store-and-forward. The one addition for today is a **radio**: instead of a
private log, each gauge `publish`es messages to a shared **bus** the cloud can read - and it `subscribe`s
to a `config` topic the cloud can write. That is the two-way street from the lecture, in code.

> **Concept - the five hybrid patterns, and the message bus.** Real edge-cloud systems are built from five reusable patterns. **Filter and forward** (the gauge distills, the cloud archives), **trigger and burst** (an edge alert summons heavy cloud compute), **train up and infer down** (the cloud learns a model, the shrunk version runs on the gauge), **adaptive command** (cloud foresight re-tunes the fleet, like storm mode), and the **gateway tier** (a middle box aggregates a cluster of gauges). We use several today. The gauges talk to the cloud over a **message bus**, our stand-in for **MQTT**, the lightweight publish-and-subscribe protocol of the internet of things: a device **publishes** tiny messages to named **topics**, and anyone interested **subscribes**, so nobody needs anybody else's network address.

In [ ]:
HW = dict(battery_mAh=10_000, sleep_mA=0.01, sensor_mA=5, sensor_s=0.05,
          mcu_mA=40, mcu_s=0.01, radio_mA=220, radio_s=2.0, bytes_per_msg=60)
mAh = lambda mA, s: mA * s / 3600.0

class MessageBus:
    """A dead-simple in-notebook MQTT stand-in: topic -> list of (time, payload)."""
    def __init__(self):
        self.log = []
    def publish(self, topic, t, payload):
        self.log.append((t, topic, payload))
    def messages(self, topic_prefix=""):
        return [(t, tp, pl) for (t, tp, pl) in self.log if tp.startswith(topic_prefix)]

class SmartGauge:
    RISE_ALERT = 1.5          # ft/hour  (config: the cloud can lower this)
    QUIET_EVERY = 15          # minutes  (config: the cloud can drop this to 1)
    def __init__(self, gid, stage, bus, network_up=None):
        self.gid, self.stage, self.bus = gid, stage, bus
        self.net = network_up if network_up is not None else np.ones(len(stage), bool)
        self.used_mAh = 0.0; self.msgs = 0; self.buffer = []
        self.hist = []; self.hour = []; self.event = False; self.flood_alarmed = False
        self.alarms = []; self._i = 0

    def read(self):
        self.used_mAh += mAh(HW["sensor_mA"], HW["sensor_s"]) + mAh(HW["mcu_mA"], HW["mcu_s"])
        return float(self.stage.iloc[self._i])

    def transmit(self, kind, value):
        self.used_mAh += mAh(HW["radio_mA"], HW["radio_s"])
        t = self.stage.index[self._i]
        if self.net[self._i]:
            self.msgs += 1
            self.bus.publish(f"basin/gauge{self.gid}/{kind}", t, {"gid": self.gid, "stage": value, "kind": kind, "measured": t})
        else:
            self.buffer.append((kind, value, t))       # store-and-forward

    def flush(self):
        while self.buffer and self.net[self._i]:
            kind, value, t = self.buffer.pop(0)
            self.used_mAh += mAh(HW["radio_mA"], HW["radio_s"]); self.msgs += 1
            self.bus.publish(f"basin/gauge{self.gid}/{kind}", self.stage.index[self._i],
                             {"gid": self.gid, "stage": value, "kind": kind + "+BACKFILL", "measured": t})

    def rise(self, t, v):
        self.hist.append((t, v)); cut = t - pd.Timedelta(hours=1)
        self.hist = [(a, b) for a, b in self.hist if a >= cut]
        return v - self.hist[0][1]

    def step(self, i, config):
        self._i = i; self.flush()
        every = 1 if self.event else self.QUIET_EVERY
        if i % every: return
        v = self.read(); self.hour.append(v); r = self.rise(self.stage.index[i], v)
        rise_thresh = config.get("rise_alert", self.RISE_ALERT)
        if (not self.event) and r > rise_thresh:
            self.event = True; self.alarms.append((self.stage.index[i], "RAPID RISE"))
            self.transmit("alert", v)
        if (not self.flood_alarmed) and v >= FLOOD_STAGE:
            self.flood_alarmed = True; self.alarms.append((self.stage.index[i], "FLOOD STAGE"))
            self.transmit("alert", v)
        if self.event and v < 8.0 and r < 0:
            self.event = False; self.flood_alarmed = False
        if i % 60 == 0 and self.hour:
            self.transmit("summary", float(np.mean(self.hour))); self.hour = []

print("Fleet class ready: adaptive sampling, 2 rules, summaries, store-and-forward, + a radio.")

---
## Part 2 — The cloud brain

The cloud subscribes to the bus. Its job is **fusion**: keep a live picture of every gauge, notice
alarms the instant they arrive, and - crucially - decide what to do about them. For now, build the
ingest-and-track half. It reads every message published so far and reports the basin state.

> **Concept - the cloud's first job is fusion, not forecasting.** Messages arrive from many gauges, some alarming, some quiet, some suddenly silent. Before it can forecast, the cloud has to decide what is actually true. **Fusion** assembles every gauge's latest reading, cross-checks the cheap dense sensors against a few trusted gauges and against physics, and treats silence as information (a gauge that goes quiet during a storm is either a dead battery or a gauge underwater, and both are urgent). This basin-wide picture is something no single gauge could ever produce, and it is why the cloud layer earns its place.

In [ ]:
bus = MessageBus()

class CloudBrain:
    def __init__(self, bus):
        self.bus = bus
        self.latest = {}          # gid -> (time, stage)
        self.alerts = []          # (time, gid, stage) upstream alarms it has seen
    def ingest(self, upto=None):
        """Read the bus up to time `upto` (or all of it). This is the cloud's live picture."""
        self.latest.clear(); self.alerts.clear()
        for t, topic, pl in self.bus.messages("basin/gauge"):
            if upto is not None and t > upto:
                continue
            self.latest[pl["gid"]] = (pl["measured"], pl["stage"])
            if "alert" in topic:
                self.alerts.append((t, pl["gid"], pl["stage"]))
        return self
    def basin_board(self):
        rows = []
        for g in range(N_GAUGES):
            if g in self.latest:
                t, v = self.latest[g]
                rows.append({"gauge": g, "last heard": t.strftime("%m-%d %H:%M"),
                             "stage_ft": round(v, 1), "state": "FLOOD" if v >= FLOOD_STAGE else "ok"})
            else:
                rows.append({"gauge": g, "last heard": "--", "stage_ft": np.nan, "state": "silent"})
        return pd.DataFrame(rows).set_index("gauge")

print("Cloud brain ready: ingest the bus, track every gauge, catch alerts.")

---
## Part 3 — Run the baseline: hybrid detection, healthy network

Run the fleet through the whole storm with the network up, feeding the cloud as messages arrive. No
storm mode yet - just filter-and-forward plus the cloud watching. We will see the alerts march
downstream gauge by gauge.

In [ ]:
def run_fleet(configs=None, network=None):
    """Simulate all gauges minute-by-minute. configs[gid] is a live-updated dict."""
    global bus
    bus = MessageBus()
    configs = configs or {g: {} for g in range(N_GAUGES)}
    nets = network or {g: np.ones(len(TIME), bool) for g in range(N_GAUGES)}
    fleet = {g: SmartGauge(g, basin[g], bus, nets[g]) for g in range(N_GAUGES)}
    for i in range(len(TIME)):
        for g in range(N_GAUGES):
            fleet[g].step(i, configs[g])
    return fleet

fleet = run_fleet()
cloud = CloudBrain(bus).ingest()

print("Alerts the cloud received, in order:")
for t, gid, v in cloud.alerts:
    print(f"  {t}   gauge {gid}   stage {v:.1f} ft")
print(f"\nTotal messages on the bus: {len(bus.log)}")

# The cloud's live picture partway through the storm - gauges 0-1 flooding, the rest still quiet.
snapshot = pd.Timestamp("2026-07-16 12:30")
print(f"\nBasin board as the cloud saw it at {snapshot}:")
CloudBrain(bus).ingest(upto=snapshot).basin_board()

**Watch the cascade.** The alert fires at gauge 0 first, then gauge 1 about 90 minutes later, and so
on down to the town. And look at the basin board: partway through the storm the cloud sees the upstream
gauges in FLOOD while the town is still calm. Each gauge, on its own, only knows its own water. The
cloud is the first entity that can see the *pattern* - an alarm marching downstream - and act on it
before the wave reaches gauge 4.

---
## Part 4 — Trigger & burst

The moment the cloud sees gauge 0's alert, it does what no microcontroller can: using the **basin
topology** - which reach flows into which, and how fast - it projects when the flood will reach *every
downstream gauge*. A single reach cannot do this; it has no idea what is upstream. This is flood
routing, and the cloud bursts it across cores the instant the alarm lands. One 60-byte message from a
$5 chip spins up the supercomputer.

> **Concept - one alert wakes the supercomputer.** This is the **trigger and burst** pattern. A 60-byte alert from a five-dollar chip is not just data, it is a starting gun. The instant it lands, the cloud grabs many CPU cores, runs a basin-wide calculation in parallel, and then releases the cores, so it idles cheaply between storms and reacts massively during one. Here the cloud uses **flood routing**: knowing the basin's shape and how fast the wave travels between gauges, it projects when the flood will reach every downstream reach. A single gauge cannot do this because it has no idea what is upstream. That is the cloud's global view turned into hours of warning for the town.

**Routing, lead time, and message savings.** When an upstream gauge alarms at $t_{\text{trig}}$, the cloud estimates the flood-stage arrival at a downstream reach from that reach's own rise rate $r_g$, or by routing along the basin at a per-hop travel time $\tau$:

$$t_{\text{flood},g}=t_0+\frac{S_{\text{flood}}-s_g(t_0)}{r_g}\qquad\text{or}\qquad t_{\text{flood},g}\approx t_{\text{trig}}+(g-g_{\text{trig}})\,\tau.$$

The warning lead time at the town and the gateway's message reduction are

$$L=t_{\text{flood}}-t_{\text{alarm}},\qquad \rho=1-\frac{U}{R},$$

where $R$ raw gauge messages become $U$ bundled uplinks.

In [ ]:
import time
from joblib import Parallel, delayed

def local_rise_rate(gid, upto, window_min=60):
    """ft/hour over the last hour of the reach's record, as the cloud has received it."""
    s = basin[gid].loc[:upto]
    recent = s[s.index >= upto - pd.Timedelta(minutes=window_min)]
    if len(recent) < 2: return 0.0
    dt_hr = (recent.index[-1] - recent.index[0]).total_seconds() / 3600
    return (recent.iloc[-1] - recent.iloc[0]) / dt_hr if dt_hr > 0 else 0.0

def route_reach(gid, trig_gid, trig_time):
    """Project this reach's flood-stage arrival from the upstream trigger + routing."""
    stage_now = float(basin[gid].loc[:trig_time].iloc[-1])
    rate = local_rise_rate(gid, trig_time)
    if stage_now >= FLOOD_STAGE:
        eta = trig_time                                   # already flooding
    elif rate > 0.3:                                      # this reach is visibly rising: extrapolate
        hrs = (FLOOD_STAGE - stage_now) / rate
        eta = trig_time + pd.Timedelta(hours=hrs)
    else:                                                 # still quiet: route from the trigger downstream
        hops = gid - trig_gid
        eta = trig_time + pd.Timedelta(minutes=hops * TRAVEL_MIN) + pd.Timedelta(hours=3.5)
    return {"gauge": gid, "stage @trigger": round(stage_now,1),
            "projected flood arrival": eta.strftime("%m-%d %H:%M"),
            "hours of notice": round((eta - trig_time).total_seconds()/3600, 1)}

trig_time, trig_gid, _ = cloud.alerts[0]   # gauge 0's first (rapid-rise) alert
print(f"TRIGGER: gauge {trig_gid} alerted at {trig_time}. Bursting basin-wide routing forecast...\n")

t0 = time.perf_counter()
burst = Parallel(n_jobs=-1)(delayed(route_reach)(g, trig_gid, trig_time) for g in range(N_GAUGES))
dt = time.perf_counter() - t0

print(f"Routed every reach in {dt:.2f}s on {__import__('os').cpu_count()} cores, then released them.\n")
pd.DataFrame(burst).set_index("gauge")

The cloud now holds something no single gauge could produce: a **basin-wide flood-arrival forecast**,
generated the instant the first alarm arrived. The town at gauge 4 learns, at 07:00, that its river will
flood in the afternoon - hours before its own water has moved a foot. That is trigger-and-burst: idle
cheaply, react massively on the event, release. The cost is pennies of credits, because the cores are
held for seconds rather than left running between storms.

---
## Part 5 — Close the loop: push storm mode down

Now the downstream direction. The cloud has seen gauge 0 alarm, so it *knows* the wave is coming for
gauges 1 to 4. It reaches down and puts those gauges into **storm mode**: sample every minute and drop
the rise threshold to a hair trigger. We rerun the fleet, but this time the cloud re-tunes each
downstream gauge the moment the wave upstream trips - and we measure the extra warning time.

> **Concept - closing the loop, and designing for failure.** So far data flowed up. Now guidance flows down: because the cloud saw the upstream alarm, it knows the wave is coming for the gauges below and pushes **storm mode** to them, lowering their triggers and speeding their sampling before their own water rises. This is the **adaptive command** pattern, and it is what makes the system more than the sum of its parts. The deeper principle behind the whole pipeline is **graceful degradation**: every layer's failure should cost breadth or convenience, never the core warning. Lose a gauge and you lose resolution, lose the link and the siren still fires, lose the whole cloud and the fleet keeps sensing and alarming on its own. A warning system must bend, not break.

In [ ]:
def run_with_storm_mode():
    """Two-pass: (1) find each gauge's natural alert, (2) rerun pushing storm mode downstream early."""
    # Pass 1: baseline alert times per gauge (no storm mode).
    base = run_fleet()
    base_alert = {}
    for g in range(N_GAUGES):
        a = [t for t, k in base[g].alarms if k == "RAPID RISE"]
        base_alert[g] = a[0] if a else None

    # Pass 2: when an upstream gauge alerts, the cloud lowers the NEXT gauge's threshold immediately.
    configs = {g: {} for g in range(N_GAUGES)}
    nets = {g: np.ones(len(TIME), bool) for g in range(N_GAUGES)}
    bus2 = MessageBus()
    fleet2 = {g: SmartGauge(g, basin[g], bus2, nets[g]) for g in range(N_GAUGES)}
    armed = set()
    for i in range(len(TIME)):
        t = TIME[i]
        for g in range(N_GAUGES):
            # cloud logic: if the gauge just above me alerted, arm storm mode on me
            if g-1 in [gg for gg in range(N_GAUGES)
                       if fleet2[gg].alarms and fleet2[gg].alarms[0][0] <= t] and g not in armed:
                configs[g] = {"rise_alert": 0.4}       # hair trigger, pushed down from the cloud
                fleet2[g].QUIET_EVERY = 1              # sample every minute now
                armed.add(g)
            fleet2[g].step(i, configs[g])
    storm_alert = {}
    for g in range(N_GAUGES):
        a = [t for t, k in fleet2[g].alarms if k == "RAPID RISE"]
        storm_alert[g] = a[0] if a else None
    return base_alert, storm_alert

base_alert, storm_alert = run_with_storm_mode()

rows = []
for g in range(N_GAUGES):
    b, s = base_alert[g], storm_alert[g]
    gained = (b - s).total_seconds()/60 if (b and s) else 0
    rows.append({"gauge": g,
                 "alert (edge-only)": b.strftime("%H:%M") if b else "--",
                 "alert (storm mode)": s.strftime("%H:%M") if s else "--",
                 "minutes gained": round(gained)})
board = pd.DataFrame(rows).set_index("gauge")
board

**Read the last column.** Gauge 0 gains nothing - it is the first to see the storm, so there is
nothing upstream to warn it. But every downstream gauge alarms **earlier** because the cloud, seeing the
alert march down the basin, lowered their triggers before their own water rose. The town at gauge 4 gets
the biggest head start. That is the hybrid payoff: **foresight the edge cannot have, delivered as
configuration the edge can act on.**

In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 4.4))
for g in range(N_GAUGES):
    ax.plot(basin[g].index, basin[g].values, lw=1.3, alpha=0.5,
            label=("town" if g==N_GAUGES-1 else ("headwaters" if g==0 else None)))
for g in range(N_GAUGES):
    if base_alert[g]:  ax.axvline(base_alert[g],  color="crimson", ls=":",  lw=1.2)
    if storm_alert[g]: ax.axvline(storm_alert[g], color="green",   ls="-",  lw=1.4)
ax.plot([], [], color="crimson", ls=":", label="edge-only alert")
ax.plot([], [], color="green", ls="-", label="storm-mode alert (earlier)")
ax.axhline(FLOOD_STAGE, color="gray", ls="--", lw=1)
ax.set_xlim(pd.Timestamp("2026-07-16 00:00"), pd.Timestamp("2026-07-17 12:00"))
ax.set_ylim(1.5, 13.5); ax.set_title("Green (hybrid) fires before red (edge-only) at every downstream gauge")
ax.set_ylabel("Stage (ft)"); ax.legend(fontsize=9, loc="upper right"); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

---
## Part 6 — The scoreboard: three architectures, one storm

Finally, the whole week on one table. We score the same flood three ways: cloud-only (Tuesday's dumb
pipe), edge-only (Wednesday's lonely hero), and today's hybrid. The metric that matters is **warning
time at the town** - and whether the warning survives an outage.

In [ ]:
# Warning time at the town (gauge 4) = flood-stage crossing minus when the town was warned.
town = N_GAUGES - 1
town_crossing = basin[town][basin[town] >= FLOOD_STAGE].index[0]

def lead_hours(alert_t):
    return round((town_crossing - alert_t).total_seconds()/3600, 1) if alert_t else None

# cloud-only: the town's flood is only "known" when a reading reaches the cloud; in an outage, never.
# edge-only: the town's own gauge alarms locally (base_alert[town]) - but no upstream foresight.
# hybrid:   the town gets storm mode from upstream (storm_alert[town]).
scoreboard = pd.DataFrame([
    {"architecture": "Cloud-only (Tue)", "warns the town": "on healthy net only",
     "town lead time": lead_hours(base_alert[town]), "survives outage": "no - flood vanishes"},
    {"architecture": "Edge-only (Wed)",  "warns the town": "local siren",
     "town lead time": lead_hours(base_alert[town]), "survives outage": "yes - siren fires"},
    {"architecture": "Hybrid (today)",   "warns the town": "siren + upstream foresight",
     "town lead time": lead_hours(storm_alert[town]), "survives outage": "yes - siren + backfill"},
]).set_index("architecture")
print(f"The town's river crosses flood stage at {town_crossing}.\n")
scoreboard

**The hybrid row wins the one number that matters** - lead time at the town - *and* keeps the
outage-survival that the edge gave us yesterday. Cloud-only has foresight but dies in the outage;
edge-only survives but has no foresight for the town; hybrid has both, because each layer did what only
it could do.

---
## Wrap-up

You assembled the whole early-warning system:

- **a fleet** of smart gauges publishing to a shared bus,
- **a cloud brain** that fused their messages into a live basin picture,
- **trigger & burst** - one edge alert summoning a basin-wide forecast in seconds,
- **a closed loop** - storm mode pushed down, buying the town extra warning time,
- and a **scoreboard** proving the hybrid beats either half alone.

### Where this goes

Every piece here is modular and yours to fork. Swap our design storm for live USGS data; add a gateway
tier; replace the rule-based detector with a TinyML model trained in the cloud and shrunk for the edge.
The architecture you just built is the same one behind IFIS and the National Water Model - you have the
skills now. Go point them at your own Week 2 question.

---
## Optional challenges (if you finish early)

1. **Longer basin.** Set `N_GAUGES = 10`. Does the town's lead time grow? Why?
2. **Cut the backhaul.** Give gauge 2 a 6-hour outage during the storm (`network=`). Does the hybrid
   still warn the town? Trace which layer carried it.
3. **Tune storm mode.** Change the pushed-down `rise_alert` from 0.4 to 0.2. More lead time, or more
   false alarms on the daily wiggle? Where is the sweet spot?
4. **Real rivers.** Replace `storm_at` with live USGS stage for five gauges down an actual Iowa river
   (use Day 2's `fetch_live_stage`). Does the cascade still show up?
5. **A gateway tier.** Add a class that sits between three gauges and the cloud, aggregating their
   messages into one uplink. How many bus messages does it save?